In [1]:
import sys
from pathlib import Path


CURRENT_PATH = Path.cwd().resolve()

if (CURRENT_PATH / "src").exists():
    PROJECT_ROOT = CURRENT_PATH
else:
    PROJECT_ROOT = CURRENT_PATH.parent


if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(PROJECT_ROOT)
    )


print(
    "Project root:",
    PROJECT_ROOT
)

Project root: D:\Internship Tasks\Task - 2026.08.19\GenAI-Powered-Customer-Support-Assistant-with-RAG


In [2]:
from src.rag import (
    retrieve_context,
    generate_rag_response,
)

print(
    "RAG module imported successfully."
)

RAG module imported successfully.


In [3]:
question = (
    "I returned my item and still "
    "have not received my refund."
)

results = retrieve_context(
    question,
    top_k=3
)

for result in results:

    print("=" * 70)

    print(
        "SOURCE:",
        result["source"]
    )

    print(
        "SCORE:",
        round(result["score"], 4)
    )

    print(
        "TEXT:"
    )

    print(
        result["text"]
    )

SOURCE: refund_policy.txt
SCORE: 0.5418
TEXT:
Refunds are normally processed within 5 to 7 business days after the returned item has been received and approved.
SOURCE: refund_policy.txt
SCORE: 0.1505
TEXT:
Refund Policy
SOURCE: refund_policy.txt
SCORE: 0.0843
TEXT:
Customers should provide their order number when requesting a refund.


In [4]:
question = (
    "I was charged twice for the same order."
)

results = retrieve_context(
    question,
    top_k=3
)

for result in results:

    print("=" * 70)

    print(
        result["source"],
        round(result["score"], 4)
    )

    print(
        result["text"]
    )

cancellation_policy.txt 0.401
Customers may request an order cancellation before the order has been shipped.
cancellation_policy.txt 0.2361
Cancellation cannot be guaranteed after an order has already been dispatched.
refund_policy.txt 0.2258
Customers should provide their order number when requesting a refund.


In [5]:
question = (
    "Please cancel my order before it ships."
)

results = retrieve_context(
    question,
    top_k=3
)

for result in results:

    print("=" * 70)

    print(
        result["source"],
        round(result["score"], 4)
    )

    print(
        result["text"]
    )

cancellation_policy.txt 0.401
Customers may request an order cancellation before the order has been shipped.
cancellation_policy.txt 0.2361
Cancellation cannot be guaranteed after an order has already been dispatched.
refund_policy.txt 0.2258
Customers should provide their order number when requesting a refund.


In [6]:
question = """
I returned my product 10 days ago
but I still have not received my refund.
"""

result = generate_rag_response(
    question
)

print(
    result["response"]
)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Hello,

Thank you for reaching out to us. 

Refunds are normally processed within 5 to 7 business days after the returned item has been received and approved. Because it has been 10 days since you returned your product, this issue requires verification. 

Please allow us some time to look into the status of your return and refund. We will follow up with you as soon as more information is available.

Best regards,
Customer Support Team


In [7]:
for item in result["retrieved_context"]:

    print("=" * 70)

    print(
        "SOURCE:",
        item["source"]
    )

    print(
        "RELEVANCE SCORE:",
        round(
            item["score"],
            4
        )
    )

    print(
        "CONTEXT:"
    )

    print(
        item["text"]
    )

SOURCE: refund_policy.txt
RELEVANCE SCORE: 0.3238
CONTEXT:
Refunds are normally processed within 5 to 7 business days after the returned item has been received and approved.
SOURCE: delivery_policy.txt
RELEVANCE SCORE: 0.2025
CONTEXT:
If an order has not arrived within 10 business days, the issue should be investigated by the delivery support team.
SOURCE: refund_policy.txt
RELEVANCE SCORE: 0.1791
CONTEXT:
Customers may request a refund within 30 days of purchase.


In [8]:
test_questions = [
    "I was charged twice for my order.",
    "My package has not arrived after 12 business days.",
    "Can I cancel my order before it ships?",
    "I cannot access my account.",
    "How long does a refund normally take?"
]

In [9]:
for question in test_questions:

    results = retrieve_context(
        question,
        top_k=1
    )

    print("=" * 80)

    print(
        "QUESTION:",
        question
    )

    print(
        "TOP POLICY:",
        results[0]["source"]
    )

    print(
        "SCORE:",
        round(
            results[0]["score"],
            4
        )
    )

QUESTION: I was charged twice for my order.
TOP POLICY: cancellation_policy.txt
SCORE: 0.401
QUESTION: My package has not arrived after 12 business days.
TOP POLICY: delivery_policy.txt
SCORE: 0.4429
QUESTION: Can I cancel my order before it ships?
TOP POLICY: cancellation_policy.txt
SCORE: 0.401
QUESTION: I cannot access my account.
TOP POLICY: account_policy.txt
SCORE: 0.457
QUESTION: How long does a refund normally take?
TOP POLICY: refund_policy.txt
SCORE: 0.3103


In [10]:
retrieval_tests = [
    {
        "question": "How long does a refund take?",
        "expected_source": "refund_policy.txt"
    },
    {
        "question": "I was charged twice.",
        "expected_source": "payment_policy.txt"
    },
    {
        "question": "My package has not arrived.",
        "expected_source": "delivery_policy.txt"
    },
    {
        "question": "Please cancel my order.",
        "expected_source": "cancellation_policy.txt"
    },
    {
        "question": "I cannot log into my account.",
        "expected_source": "account_policy.txt"
    }
]

In [11]:
evaluation_results = []

for test in retrieval_tests:

    result = retrieve_context(
        test["question"],
        top_k=1
    )[0]

    predicted_source = result["source"]

    correct = (
        predicted_source
        ==
        test["expected_source"]
    )

    evaluation_results.append({
        "question": test["question"],
        "expected_source": test["expected_source"],
        "predicted_source": predicted_source,
        "correct": correct,
        "score": result["score"]
    })

In [12]:
import pandas as pd


retrieval_df = pd.DataFrame(
    evaluation_results
)

retrieval_df

,question,expected_source,predicted_source,correct,score
0,How long does a refund take?,refund_policy.txt,refund_policy.txt,True,0.499721
1,I was charged twice.,payment_policy.txt,refund_policy.txt,False,0.000000
2,My package has not arrived.,delivery_policy.txt,delivery_policy.txt,True,0.260204
3,Please cancel my order.,cancellation_policy.txt,cancellation_policy.txt,True,0.400977
4,I cannot log into my account.,account_policy.txt,account_policy.txt,True,0.383397


In [13]:
retrieval_accuracy = (
    retrieval_df["correct"].mean()
)

print(
    f"Retrieval Accuracy: "
    f"{retrieval_accuracy:.2%}"
)

Retrieval Accuracy: 80.00%


In [14]:
retrieval_df.to_csv(
    "../evaluation/rag_retrieval_results.csv",
    index=False
)